In [ ]:
%matplotlib qt

import json

import hyperspy.api as hs
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection

from espm.conf import SYMBOLS_PERIODIC_TABLE
from espm.datasets.eds_spim import gaussian

this is to set up the matplotlib plot, please ignore

In [ ]:
with open(SYMBOLS_PERIODIC_TABLE, "r") as f:
    SPT = json.load(f)["table"]

cmap = plt.get_cmap("tab10")


def plot_table(ax, table, elements, energy_axis, l1="--", l2="-.", bell=False):
    xs = []
    colours = []
    linestyles = []

    legend = {}

    bell_segments = []
    bell_colours = []

    hover_data = []

    idx = set()

    for i, elt in enumerate(elements):
        elt_num_str = str(SPT[elt]["number"])
        db_entries = table[elt_num_str]

        colour = cmap(i)
        legend[elt] = colour

        for name, data in db_entries.items():
            energy = data["energy"]

            xs.append(energy)
            colours.append(colour)
            linestyles.append(l1 if energy not in idx else l2)

            hover_data.append({"x": energy, "name": f"{elt} {name}", "color": colour})

            if bell:
                sigma = data["sigma"]
                A = data["amplitude"]
                C = data["background_base"]
                m = data["background_slope"]

                mask = (energy_axis > energy - 3 * sigma) & (
                    energy_axis < energy + 3 * sigma
                )
                x_axis = energy_axis[mask]

                y_gauss = gaussian(x_axis, A, energy, sigma, C, m)

                points = np.column_stack([x_axis, y_gauss])
                bell_segments.append(points)
                bell_colours.append(colour)

            idx.add(energy)

    vline_collection = ax.vlines(
        x=xs,
        ymin=-0.1,
        ymax=2,
        colors=colours,
        linestyles=linestyles,
        linewidths=1,
        pickradius=5,
    )

    if bell:
        bell_collection = LineCollection(
            bell_segments, colors=bell_colours, linewidths=1.5
        )
        ax.add_collection(bell_collection)

    for elt, col in legend.items():
        ax.plot([], [], color=col, label=elt, lw=1)
    ax.legend()

    annot = ax.annotate(
        "",
        xy=(0, 0),
        xytext=(10, 10),
        textcoords="offset points",
        bbox=dict(boxstyle="round", fc="w", alpha=0.9, ec="gray"),
    )
    annot.set_visible(False)

    def on_hover(event):
        if event.inaxes == ax:
            contained, info = vline_collection.contains(event)

            if contained:
                line_idx = info["ind"][0]
                item = hover_data[line_idx]

                annot.xy = (item["x"], event.ydata)
                annot.set_text(f"{item['name']}: {item['x']:.3f}")
                annot.get_bbox_patch().set_edgecolor(item["color"])

                if not annot.get_visible():
                    annot.set_visible(True)
                    fig.canvas.draw_idle()
                return

        if annot.get_visible():
            annot.set_visible(False)
            fig.canvas.draw_idle()

    def on_press(event):
        if event.key == " ":
            toolbar = event.canvas.toolbar
            if toolbar.mode != "pan/zoom":
                toolbar.pan()

    def on_release(event):
        if event.key == " ":
            toolbar = event.canvas.toolbar
            if toolbar.mode == "pan/zoom":
                toolbar.pan()

    fig = ax.figure
    fig.canvas.mpl_connect("motion_notify_event", on_hover)
    fig.canvas.mpl_connect("key_press_event", on_press)
    fig.canvas.mpl_connect("key_release_event", on_release)


def plot(funcs, title):
    _, ax = plt.subplots()

    for func in funcs:
        func(ax)

    ax.set_title(title)
    ax.set_xlabel("Energy (keV)")
    ax.set_ylabel("Intensity (counts)")

    plt.show()


def plot_avg(ax, energy_axis, avg_spectrum):
    ax.plot(energy_axis, avg_spectrum, color="k", label="input", linewidth=3)

In [ ]:
filename = "X3-13MAY22_MAP06.bcf"

signals = hs.load(filename)
eds_sig = signals[1].isig[0.1:]
eds_sig.set_signal_type("EDS_espm")

eds_sig.set_analysis_parameters(
    thickness=10e-5,
    density=4.1,
    detector_type="SDD_efficiency.txt",
    width_slope=0.01,
    width_intercept=0.065,
    geom_eff=None,
    xray_db="200keV_xrays.json",
)

eds_sig.metadata.Sample.elements

In [ ]:
average_spectrum = eds_sig.mean(axis=(0, 1)).data
energy_axis = eds_sig.axes_manager.signal_axes[0].axis
elements = eds_sig.metadata.Sample.elements
# elements.remove("Re")
theoretical = eds_sig.model.db_dict

plot theoretical lines

In [ ]:
plot(
    [
        lambda ax: plot_avg(ax, energy_axis, average_spectrum),
        lambda ax: plot_table(ax, theoretical, elements, energy_axis),
    ],
    "theoretical lines",
)

plot theoretical lines vs individual peak fit

In [ ]:
plot(
    [
        lambda ax: plot_avg(ax, energy_axis, average_spectrum),
        lambda ax: plot_table(ax, theoretical, elements, energy_axis),
        lambda ax: plot_table(
            ax,
            eds_sig.fit_table(),
            elements,
            energy_axis,
            l1=":",
            l2="-",
            bell=True,
        ),
    ],
    "theoretical vs peak fit",
)

plot the polynomial that maps theoretical energy to calibrated energy.

In [ ]:
# Change this to see weighted or unweighted polynomial fits.
WEIGHTED = True

fig, ax = plt.subplots()

lines = []
labels = []

for d in range(1, 20):
    _, y_poly, _ = eds_sig.poly_fit(degree=d, weighted=WEIGHTED)
    ax.plot(energy_axis, np.polyval(y_poly, energy_axis), label=f"degree {d}")

ax.plot(energy_axis, energy_axis, label="identity", color="k", linewidth=3, zorder=50)

x = []
y = []
c = []

calibrated_peaks = eds_sig.fit_table()

for i, el in enumerate(calibrated_peaks):
    lines = calibrated_peaks[el]
    for _, line in lines.items():
        x.append(line["theoretical"])
        y.append(line["energy"])
        c.append(cmap(i))

ax.scatter(x, y, c=c, zorder=100)

ax.set_aspect("equal")
ax.set_box_aspect(1)

ax.legend(loc="upper right")

ax.set_title("theoretical vs calibrated energy")
ax.set_xlabel("Theoretical Energy (keV)")
ax.set_ylabel("Calibrated Energy (keV)")

ax.set_xlim(0, 20)
ax.set_ylim(0, 20)

plt.show()

theoretical lines vs poly fitted lines (unweighted)

In [ ]:
# Change this to see different polynomial degree
DEGREE = 2

plot(
    [
        lambda ax: plot_avg(ax, energy_axis, average_spectrum),
        lambda ax: plot_table(ax, theoretical, elements, energy_axis),
        lambda ax: plot_table(
            ax,
            eds_sig.poly_fit(degree=DEGREE)[0],
            elements,
            energy_axis,
            l1=":",
            l2="-",
            bell=True,
        ),
    ],
    title=f"poly fit deg {DEGREE} (unweighted)",
)

theoretical lines vs poly fitted lines (weighted)

In [ ]:
# Change this to see different polynomial degree
DEGREE = 2

plot(
    [
        lambda ax: plot_avg(ax, energy_axis, average_spectrum),
        lambda ax: plot_table(ax, theoretical, elements, energy_axis),
        lambda ax: plot_table(
            ax,
            eds_sig.poly_fit(degree=DEGREE, weighted=True)[0],
            elements,
            energy_axis,
            l1=":",
            l2="-",
            bell=True,
        ),
    ],
    title=f"poly fit deg {DEGREE} (weighted)",
)